# 03-4: RandomForest モデル

M5 Forecasting - Accuracy コンペティション

- **モデル**: RandomForest (Ensemble)
- **評価指標**: RMSE / RMSLE
- **NaN戦略**: **中央値埋め + 欠損フラグ** — RFはNaN非対応（Lesson 16-18）
  - `fillna(-999)` は過学習の主因 → 使わない
  - 中央値は分布に馴染む → 不自然な分割を回避
  - 欠損フラグで情報保持
- **CV戦略**: 時系列5-fold CV（28日window、全モデル統一）
- **max_depth**: 20に制限（Lesson 18: 深い木は過学習を助長）

In [ ]:
import sys
import warnings
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

warnings.filterwarnings('ignore')
plt.rcParams['font.family'] = 'MS Gothic'
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['axes.unicode_minus'] = False

sys.path.append('../../..')
sys.path.append('../..')
from src.features import build_features, prepare_train_data, get_val_folds, get_feature_cols

INTERMEDIATE_DIR = Path('./intermediate')

def rmsle(y_true, y_pred):
    y_pred = np.clip(y_pred, 0, None)
    return np.sqrt(mean_squared_error(np.log1p(y_true), np.log1p(y_pred)))

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


def impute_with_median_and_flags(X_train, X_valid, features):
    """NaN非対応モデル用: 中央値埋め + 欠損フラグ列の追加（Lesson 16-17）

    - 中央値は特徴量の分布に馴染む → 不自然な分割が発生しない
    - 欠損フラグで「欠損だった」情報を保持 → 情報損失なし
    - 検証データにも学習データの中央値を使用 → リーク防止
    """
    X_train = X_train.copy()
    X_valid = X_valid.copy()
    medians = X_train[features].median()
    nan_flag_cols = []
    for col in features:
        if X_train[col].isna().any() or X_valid[col].isna().any():
            flag_col = f'is_nan_{col}'
            X_train[flag_col] = X_train[col].isna().astype(int)
            X_valid[flag_col] = X_valid[col].isna().astype(int)
            nan_flag_cols.append(flag_col)
        X_train[col] = X_train[col].fillna(medians[col])
        X_valid[col] = X_valid[col].fillna(medians[col])
    return X_train, X_valid, nan_flag_cols


print('セットアップ完了')

## 1. データ読み込みと特徴量生成

In [ ]:
df, feature_cols, val_folds = build_features(use_cache=True)
df_train = prepare_train_data(df, feature_cols)

# M5データは巨大なため、RFではサンプリングして学習
# 全データ(~5000万行)ではRF 200本のCV学習は非現実的
# → 直近の学習データに絞る or サンプリング
SAMPLE_FRAC = 0.1  # 10%サンプリング（メモリ/時間制約）
print(f'特徴量数: {len(feature_cols)}')
print(f'サンプリング率: {SAMPLE_FRAC*100:.0f}%')
print(f'サンプリング前: {len(df_train):,} rows')

df_train_sampled = df_train.sample(frac=SAMPLE_FRAC, random_state=42).reset_index(drop=True)
print(f'サンプリング後: {len(df_train_sampled):,} rows')

## 2. RandomForest パラメータ設定

In [ ]:
rf_params = {
    'n_estimators': 200,
    'max_depth': 20,          # Lesson 18: 深い木は過学習を助長 → 20に制限
    'min_samples_split': 50,
    'min_samples_leaf': 20,
    'max_features': 0.8,
    'random_state': 42,
    'n_jobs': -1,
    'verbose': 0,
}

print('RandomForest パラメータ:')
for k, v in rf_params.items():
    print(f'  {k}: {v}')

## 3. 時系列CV評価（中央値埋め + 欠損フラグ）

In [ ]:
cv_scores = []
importances = []

for fold_info in val_folds:
    fold = fold_info['fold']
    val_start = pd.Timestamp(fold_info['val_start'])
    val_end = pd.Timestamp(fold_info['val_end'])
    
    # サンプリング済みデータで分割
    train_mask = df_train_sampled['date'] < val_start
    valid_mask = (df_train_sampled['date'] >= val_start) & (df_train_sampled['date'] <= val_end)
    
    X_tr = df_train_sampled.loc[train_mask, feature_cols]
    y_tr = df_train_sampled.loc[train_mask, 'sales']
    X_va = df_train_sampled.loc[valid_mask, feature_cols]
    y_va = df_train_sampled.loc[valid_mask, 'sales']
    
    if len(X_va) == 0:
        print(f'\n--- Fold {fold}: 検証データなし（スキップ） ---')
        continue
    
    print(f'\n--- Fold {fold} ---')
    print(f'  Train: {X_tr.shape[0]:,} rows, Valid: {X_va.shape[0]:,} rows')
    
    # Lesson 16-17: 中央値埋め + 欠損フラグ（fold学習データの中央値を使用）
    X_tr_imp, X_va_imp, nan_flag_cols = impute_with_median_and_flags(X_tr, X_va, feature_cols)
    all_features = feature_cols + nan_flag_cols
    
    if fold == 1:
        print(f'  欠損フラグ列: {nan_flag_cols}')
        print(f'  全特徴量数: {len(all_features)} (元{len(feature_cols)} + フラグ{len(nan_flag_cols)})')
    
    # 学習
    model = RandomForestRegressor(**rf_params)
    model.fit(X_tr_imp[all_features], y_tr)
    
    # 予測 & 評価
    pred = model.predict(X_va_imp[all_features])
    pred = np.clip(pred, 0, None)
    score = rmse(y_va, pred)
    score_rmsle = rmsle(y_va, pred)
    
    cv_scores.append({
        'fold': fold, 'rmse': score, 'rmsle': score_rmsle,
    })
    
    # 特徴量重要度
    imp = pd.DataFrame({
        'feature': all_features,
        'importance': model.feature_importances_,
    })
    imp['fold'] = fold
    importances.append(imp)
    
    # V11: 過学習診断
    train_pred = np.clip(model.predict(X_tr_imp[all_features]), 0, None)
    train_score = rmse(y_tr, train_pred)
    ratio = train_score / score
    print(f'  RMSE: {score:.4f}, RMSLE: {score_rmsle:.5f}')
    print(f'  Train RMSE: {train_score:.4f}, 過学習比率: {ratio:.3f}')

cv_df = pd.DataFrame(cv_scores)
print(f'\n=== CV結果サマリー ===')
print(f'RMSE  平均: {cv_df["rmse"].mean():.4f} ± {cv_df["rmse"].std():.4f}')
print(f'RMSLE 平均: {cv_df["rmsle"].mean():.5f} ± {cv_df["rmsle"].std():.5f}')
print(cv_df.to_string(index=False))

## 4. 特徴量重要度

In [ ]:
imp_all = pd.concat(importances)
imp_mean = imp_all.groupby('feature')['importance'].mean().sort_values(ascending=False).reset_index()

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(imp_mean['feature'][::-1], imp_mean['importance'][::-1], color='purple')
ax.set_title('RandomForest 特徴量重要度（平均）')
ax.set_xlabel('平均重要度')
plt.tight_layout()
plt.show()

## 5. 中間データ保存

In [ ]:
# 過学習比率の平均
avg_overfit = np.mean([cv_scores[i].get('overfit_ratio', 0) for i in range(len(cv_scores))] or [0])

results = {
    'model_name': 'RandomForest',
    'params': rf_params,
    'nan_strategy': '中央値埋め + 欠損フラグ (Lesson 16-17)',
    'sample_frac': SAMPLE_FRAC,
    'cv_scores': cv_df.to_dict('records'),
    'cv_mean_rmse': cv_df['rmse'].mean(),
    'cv_std_rmse': cv_df['rmse'].std(),
    'cv_mean_rmsle': cv_df['rmsle'].mean(),
    'cv_std_rmsle': cv_df['rmsle'].std(),
    'feature_importance': imp_mean.to_dict('records'),
    'nan_flag_cols': nan_flag_cols,
}

with open(INTERMEDIATE_DIR / '03-4_rf_results.pkl', 'wb') as f:
    pickle.dump(results, f)

print('03-4_rf_results.pkl を保存しました')
print(f'\n最終結果: CV RMSE = {cv_df["rmse"].mean():.4f} ± {cv_df["rmse"].std():.4f}')
print(f'注意: {SAMPLE_FRAC*100:.0f}%サンプリングでの結果。全データではスコアが改善する可能性あり。')